In [ ]:
# --- Setup & Config ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pickle
import os
import sys
from scipy.stats import norm, ttest_ind, skew, kurtosis
from scipy.optimize import curve_fit

sys.path.append(os.path.abspath('data'))
from npj_style_v1 import set_npj_style, add_panel_label

output_dir = 'figs'
os.makedirs(output_dir, exist_ok=True)
print(f'GLOBAL output_dir set to: {output_dir}')

SAVE_DPI = 600

DATA_PATH = 'data'


In [ ]:
# --- Figure 1: RDF ---
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os
import sys
sys.path.append(os.path.abspath('data'))
from npj_style_v1 import set_npj_style, add_panel_label

os.makedirs(output_dir, exist_ok=True)

width, _ = set_npj_style(column_type='double')
height = width * 0.4 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width, height), constrained_layout=True)

with open(os.path.join(DATA_PATH, 'dataset.pkl'), 'rb') as f:
    dataset = pickle.load(f)
    db = dataset['analysis']

# --- Panel a: Crosslinker-Crosslinker RDF ---
target_dp = 'DP_30'
colors = {'PDMS': '#0072B2', 'PMTFPS': '#D55E00'} 

if target_dp in db['PDMS']:
    bins_c, gr_c = db['PDMS'][target_dp]['rdf_crystal']
    if bins_c is not None:
        ax1.plot(bins_c, gr_c, color='gray', linestyle='--', linewidth=0.5, 
                 alpha=0.4, label='Initial lattice')

for chem in ['PDMS', 'PMTFPS']:
    if target_dp in db[chem]:
        bins, gr = db[chem][target_dp]['rdf_amorphous']
        if bins is not None:
            label_text = f"{chem} (equilibrated)"
            ax1.plot(bins, gr, lw=1.0, color=colors.get(chem, 'k'), label=label_text)

ax1.set_xlabel(r'Distance $r$ ($\mathrm{\AA}$)')
ax1.set_ylabel(r'Radial distribution $g(r)$') # Updated Y-label
ax1.set_xlim(0, 50)
ax1.legend(frameon=False, loc='upper right')

ax1.text(0.05, 0.9, r'$DP=30$', transform=ax1.transAxes, fontsize=8)

# --- Panel b: Effect of Chain Length ---
chem = 'PDMS'
dp_colors = {'DP_10': '#E69F00', 'DP_30': '#CC79A7', 'DP_100': '#0072B2'}

for dp in ['DP_10', 'DP_30', 'DP_100']:
    if dp in db[chem]:
        bins, gr = db[chem][dp]['rdf_amorphous']
        if bins is not None:
            dp_val = dp.split('_')[1]
            label_clean = fr'$DP={dp_val}$'
            ax2.plot(bins, gr, lw=1.0, color=dp_colors.get(dp, 'k'), label=label_clean)

ax2.set_xlabel(r'Distance $r$ ($\mathrm{\AA}$)')
ax2.set_ylabel(r'Radial distribution $g(r)$') # Updated Y-label
ax2.set_xlim(0, 50)
ax2.set_ylim(-0.25, 2.25)
ax2.legend(frameon=False, loc='upper right')

ax2.text(0.05, 0.9, 'PDMS', transform=ax2.transAxes, fontsize=8)

# --- Zero-Line Alignment ---
y2_min, y2_max = ax2.get_ylim()
zero_ratio = (0 - y2_min) / (y2_max - y2_min)
y1_min, y1_max = ax1.get_ylim()
new_y1_min = - (zero_ratio * y1_max) / (1 - zero_ratio)
ax1.set_ylim(new_y1_min, y1_max)

# --- Alignment Logic ---
fig.canvas.draw()

def align_label_to_ylabel(ax, label_text):
    ylabel = ax.yaxis.label
    bbox = ylabel.get_window_extent(renderer=fig.canvas.get_renderer())
    ylabel_center_x = (bbox.x0 + bbox.x1) / 2

    ax_bbox = ax.get_window_extent()
    ax_width = ax_bbox.width

    x_rel = (ylabel_center_x - ax_bbox.x0) / ax_width

    ax.text(x_rel, 1.0, label_text, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='bottom', ha='center')

align_label_to_ylabel(ax1, 'a')
align_label_to_ylabel(ax2, 'b')

ax1.axhline(1, color='black', linewidth=0.5, alpha=0.1, zorder=0)
ax2.axhline(1, color='black', linewidth=0.5, alpha=0.1, zorder=0)

# plt.savefig(os.path.join(output_dir, 'Figure_1_RDF.pdf'), dpi=SAVE_DPI)
plt.savefig(os.path.join(output_dir, 'Figure_1_RDF.png'), dpi=SAVE_DPI)
print("Figure 1 re-generated with dynamic label alignment.")

In [ ]:
# --- Figure 2: Scaling ---
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os
import sys
from scipy.stats import norm

sys.path.append(os.path.abspath('data'))
from npj_style_v1 import set_npj_style

os.makedirs(output_dir, exist_ok=True)

width, _ = set_npj_style(column_type='double')
height = width * 0.40 

plt.rcParams['figure.constrained_layout.use'] = False

fig = plt.figure(figsize=(width, height), constrained_layout=False)
import matplotlib.gridspec as gridspec
gs = gridspec.GridSpec(2, 2, figure=fig, width_ratios=[1, 1], height_ratios=[1, 1], 
                       wspace=0.3, hspace=0)

ax_scaling = fig.add_subplot(gs[:, 0])      # Left: Scaling
ax_dist_top = fig.add_subplot(gs[0, 1])     # Right Top: PDMS Dist
ax_dist_bot = fig.add_subplot(gs[1, 1], sharex=ax_dist_top) # Right Bot: FPDMS Dist

with open(os.path.join(DATA_PATH, 'dataset.pkl'), 'rb') as f:
    dataset = pickle.load(f)
db = dataset['analysis']

# --- CONFIG ---
dps = ['DP_10', 'DP_30', 'DP_100']
colors = {'DP_10': '#E69F00', 'DP_30': '#CC79A7', 'DP_100': '#0072B2'}

def get_data(chem, dp):
    if chem in db and dp in db[chem]:
        try:
            return db[chem][dp]['ree_data'], db[chem][dp]['rg_data']
        except (KeyError, TypeError):
             return [], []
    return [], []

# =============================================================================
# =============================================================================
chem_styles = {'PDMS': {'c': '#0072B2', 'fmt': 'o'}, 'PMTFPS': {'c': '#D55E00', 'fmt': 's'}}

handles_data = []
handles_fits = []

for chem in ['PDMS', 'PMTFPS']:
    N_vals = []
    R2_vals = []
    for dp in dps:
        ree_vals, _ = get_data(chem, dp)
        if len(ree_vals) > 0:
            actual_N = int(dp.split('_')[1])
            N_vals.append(actual_N)
            R2_vals.append(np.mean(ree_vals**2))

    if len(N_vals) > 0:
        style = chem_styles[chem]
        h_data, = ax_scaling.loglog(N_vals, R2_vals, style['fmt'], color=style['c'], label=f'{chem}', markersize=6)
        handles_data.append(h_data)

        if len(N_vals) > 1:
            log_N = np.log(N_vals)
            log_R2 = np.log(R2_vals)
            (slope, intercept), cov = np.polyfit(log_N, log_R2, 1, cov=True)
            nu = slope / 2

            err_slope = np.sqrt(np.diag(cov))[0]
            err_nu = err_slope / 2

            x_fit = np.linspace(min(N_vals)*0.9, max(N_vals)*1.1, 10)
            y_fit = np.exp(intercept) * x_fit**slope
            h_fit, = ax_scaling.loglog(x_fit, y_fit, '-', color=style['c'], alpha=0.5, lw=1.5, 
                       label=fr'$\mathit{{\nu}}={nu:.2f}\pm{err_nu:.2f}$')
            handles_fits.append(h_fit)

from matplotlib.patches import Rectangle
h_spacer = Rectangle((0,0), 1, 1, fill=False, edgecolor='none', visible=False, label='')

final_handles = handles_data + [h_spacer] + handles_fits
ax_scaling.legend(handles=final_handles, frameon=False, loc='lower right', ncol=1)

tri_x_start = 40 
tri_y_start = 12 * 40**(1.0) * 4.5 # nudge up (3.5 -> 4.5)
tri_size = 1.6 

x0, y0 = tri_x_start, tri_y_start
x1 = tri_x_start * tri_size
y1 = tri_y_start * tri_size**1.0 # Slope 1

tx = [x0, x0, x1, x0]
ty = [y0, y1, y1, y0]

ax_scaling.loglog(tx, ty, 'k-', lw=1.0, alpha=0.8)

label_x = np.sqrt(x0 * x1)
label_y = y1 * 1.05 # Lowered (1.15 -> 1.05)
ax_scaling.text(label_x, label_y, r'$\mathit{\nu}=0.5$', 
                fontsize=7, ha='center', va='bottom')

ax_scaling.grid(True, which='major', linestyle='--', alpha=0.15, linewidth=0.5)

import matplotlib.ticker as ticker
ax_scaling.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
ax_scaling.set_yticks([300, 1000, 3000])
ax_scaling.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
ax_scaling.set_ylim(250, 4800) 

ax_scaling.set_xticks([10, 30, 100])
ax_scaling.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax_scaling.set_xlim(left=8.1)

ax_scaling.minorticks_on()
minor_subs = [2, 3, 4, 5, 6, 7, 8, 9]
ax_scaling.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=minor_subs, numticks=100))
ax_scaling.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=minor_subs, numticks=100))
ax_scaling.tick_params(which='minor', length=3, color=(0,0,0,0.5), width=0.5)

ax_scaling.set_xlabel(r'Degree of polymerization ($DP$)') # Fixed N -> DP
ax_scaling.set_ylabel(r'$\langle R_{ee}^2 \rangle$ ($\mathrm{\AA}^2$)')

# =============================================================================
# =============================================================================

def plot_distributions(ax, chem_name, hide_xlabel=False):
    ax.text(0.05, 0.8, chem_name, transform=ax.transAxes, fontsize=8)

    for dp in dps:
        ree_data, _ = get_data(chem_name, dp)
        if len(ree_data) > 0:
            dp_val = dp.split('_')[1]
            label_clean = fr'$DP={dp_val}$'

            ax.hist(ree_data, bins=30, density=True, histtype='step', 
                    linewidth=1.0, color=colors[dp], alpha=0.6)

            mu, std = norm.fit(ree_data)
            xmin, xmax = ax.get_xlim()
            if xmax <= 1.0: xmin, xmax = 0, max(ree_data)*1.1
            x = np.linspace(0, xmax, 100)
            p = norm.pdf(x, mu, std)
            ax.plot(x, p, '-', linewidth=1.5, color=colors[dp], label=label_clean)

    ax.set_yticks([0, 0.05, 0.1, 0.15])
    ax.set_yticklabels(['0', '0.05', '0.1', '0.15'])
    ax.set_ylabel(r'$P(R_{ee})$') # Updated per user request
    ax.set_ylim(0, 0.18)

    if hide_xlabel:
        ax.tick_params(axis='x', labelbottom=False) 
    else:
        ax.set_xlabel(r'$R_{ee}$ ($\mathrm{\AA}$)')
        ax.set_xticks([0, 40, 80, 120])
        ax.set_xticklabels(['0', '40', '80', '120'])
        ax.set_xlim(0, 130)

plot_distributions(ax_dist_top, 'PDMS', hide_xlabel=True)
ax_dist_top.legend(frameon=False, loc='upper right')

plot_distributions(ax_dist_bot, 'PMTFPS', hide_xlabel=False)

# --- Alignment Logic ---
fig.canvas.draw()

def align_label_to_ylabel(ax, label_text):
    ylabel = ax.yaxis.label
    bbox = ylabel.get_window_extent(renderer=fig.canvas.get_renderer())
    ylabel_center_x = (bbox.x0 + bbox.x1) / 2

    ax_bbox = ax.get_window_extent()
    ax_width = ax_bbox.width

    x_rel = (ylabel_center_x - ax_bbox.x0) / ax_width

    ax.text(x_rel, 1.0, label_text, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='bottom', ha='center')

align_label_to_ylabel(ax_scaling, 'a')
align_label_to_ylabel(ax_dist_top, 'b')

# plt.savefig(os.path.join(output_dir, 'Figure_2_Scaling_and_Distributions.pdf'), dpi=SAVE_DPI)
plt.savefig(os.path.join(output_dir, 'Figure_2_Scaling_and_Distributions.png'), dpi=SAVE_DPI)
print("Figure 2 generated successfully.")

In [ ]:
# --- Figure 3: Characteristic Ratio ---
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os
import sys

sys.path.append(os.path.abspath('data'))
from npj_style_v1 import set_npj_style

os.makedirs(output_dir, exist_ok=True)

width, _ = set_npj_style(column_type='double')
height = width * 0.4 

plt.rcParams['figure.constrained_layout.use'] = True
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width, height), constrained_layout=True)

# ---------------------------------------------------------
# ---------------------------------------------------------
def generate_semiflexible_walks(n_samples, n_steps, c_inf=1.0):
    vecs = np.zeros((n_samples, int(n_steps), 3))
    v0 = np.random.randn(n_samples, 3)
    v0 /= np.linalg.norm(v0, axis=1, keepdims=True)
    vecs[:, 0, :] = v0

    if c_inf <= 1.0 + 1e-5:
        for i in range(1, int(n_steps)):
            v = np.random.randn(n_samples, 3)
            v /= np.linalg.norm(v, axis=1, keepdims=True)
            vecs[:, i, :] = v
    else:
        cos_theta = (c_inf - 1) / (c_inf + 1)
        sin_theta = np.sqrt(1 - cos_theta**2)

        for i in range(1, int(n_steps)):
            w = vecs[:, i-1, :]
            phi = np.random.uniform(0, 2*np.pi, n_samples)
            r = np.random.randn(n_samples, 3)
            u = np.cross(r, w)
            u_norms = np.linalg.norm(u, axis=1, keepdims=True)

            mask_zero = u_norms.flatten() < 1e-6
            if np.any(mask_zero):
                u[mask_zero] = np.cross([1,0,0], w[mask_zero])
                u_norms[mask_zero] = np.linalg.norm(u[mask_zero], axis=1, keepdims=True)

            u /= u_norms
            v = np.cross(w, u)

            step = sin_theta * (np.cos(phi)[:, None] * u + np.sin(phi)[:, None] * v) + cos_theta * w
            step /= np.linalg.norm(step, axis=1, keepdims=True)
            vecs[:, i, :] = step

    return vecs

def simulate_bridge_ensemble(N_steps, target_Ree_b, c_inf=1.0, tolerance=0.15, n_samples=20000):
    vecs = generate_semiflexible_walks(n_samples, N_steps, c_inf=c_inf)
    paths = np.zeros((n_samples, int(N_steps)+1, 3))
    paths[:, 1:, :] = np.cumsum(vecs, axis=1)

    ree_vectors = paths[:, -1, :] - paths[:, 0, :]
    ree_dists = np.linalg.norm(ree_vectors, axis=1)

    active_tol = tolerance
    mask = (ree_dists >= target_Ree_b * (1 - active_tol)) & \
           (ree_dists <= target_Ree_b * (1 + active_tol))

    if np.sum(mask) < 200:
        active_tol = 0.25
        mask = (ree_dists >= target_Ree_b * (1 - active_tol)) & \
               (ree_dists <= target_Ree_b * (1 + active_tol))

    if np.sum(mask) < 10: return None, None

    constrained_paths = paths[mask]
    dists_sq = np.sum(constrained_paths[:, 1:, :]**2, axis=2)
    mean_dists_sq = np.mean(dists_sq, axis=0)
    n_axis = np.arange(1, int(N_steps) + 1)

    return n_axis, mean_dists_sq / n_axis

# ---------------------------------------------------------
# ---------------------------------------------------------
with open(os.path.join(DATA_PATH, 'dataset.pkl'), 'rb') as f:
    dataset = pickle.load(f)
    db = dataset['analysis']

dps = ['DP_10', 'DP_30', 'DP_100']
dps = ['DP_10', 'DP_30', 'DP_100']
colors = {'DP_10': '#E69F00', 'DP_30': '#CC79A7', 'DP_100': '#0072B2'} 
fit_colors = {'DP_10': '#a16f00', 'DP_30': '#8f5575', 'DP_100': '#00507d'} 
manual_cinf_pdms = {'DP_10': 4.5, 'DP_30': 9.0, 'DP_100': 9.25}
manual_cinf_ptfpms = {'DP_10': 4.5, 'DP_30': 8.0, 'DP_100': 8.5}

# ---------------------------------------------------------
# ---------------------------------------------------------
def plot_bridge_panel(ax, chem_name, stiffness_dict):
    max_y = 0
    for dp in dps:
        if dp in db[chem_name]:
            n_data, msid = db[chem_name][dp]['msid_data']
            ree_data = db[chem_name][dp]['ree_data']

            if n_data is not None:
                b = np.sqrt(msid[0]) if msid[0] > 0 else 1.5
                cn_data = msid / (n_data * b**2)
                max_y = max(max_y, np.max(cn_data))

                dp_val = dp.split('_')[1]
                label_latex = fr'$DP={dp_val}$'

                ax.plot(n_data, cn_data, 'o', markersize=2.5, alpha=0.7, markeredgewidth=0.1,
                        color=colors[dp], label=label_latex)

                rms_ree_b = np.sqrt(np.mean(ree_data**2)) / b
                user_c = stiffness_dict[dp]

                s_n, s_cn = simulate_bridge_ensemble(n_data[-1], rms_ree_b, 
                                                     c_inf=user_c, n_samples=40000) 

                if s_n is not None:
                    ax.plot(s_n, s_cn, '-', lw=1.5, alpha=0.5, color=fit_colors.get(dp, colors[dp]))

    ax.set_xlabel(r'Sub-chain index $n$') # Keeps n, but ensures font matches
    return max_y

ymax1 = plot_bridge_panel(ax1, 'PDMS', manual_cinf_pdms)
ymax2 = plot_bridge_panel(ax2, 'PMTFPS', manual_cinf_ptfpms)

global_max = max(ymax1, ymax2)
ax1.set_ylim(0, global_max * 1.2)
ax2.set_ylim(0, global_max * 1.2)

ax1.set_ylabel(r'Characteristic ratio $C_n$')
ax2.set_ylabel(r'Characteristic ratio $C_n$') 

# ---------------------------------------------------------
# ---------------------------------------------------------

ax1.text(0.05, 0.9, 'PDMS', transform=ax1.transAxes, fontsize=8)
ax2.text(0.05, 0.9, 'PMTFPS', transform=ax2.transAxes, fontsize=8)

from matplotlib.lines import Line2D

style_handles = [
    Line2D([0], [0], marker='o', linestyle='none', color='gray',
           markersize=3, markeredgewidth=0.1, label='MD simulation'),
    Line2D([0], [0], color='gray', lw=1.5, alpha=0.7,
           label='Semi-flexible Brownian bridge'),
]
color_handles = [
    Line2D([0], [0], color=colors[dp], lw=2,
           label=fr'$DP={dp.split("_")[1]}$') for dp in dps
]

leg_style = ax1.legend(handles=style_handles, frameon=False,
                       loc='lower right', fontsize=6)
ax1.add_artist(leg_style)
ax1.legend(handles=color_handles, frameon=False,
           loc='upper right', fontsize=6)

def align_label_to_ylabel(ax, label_text):
    fig.canvas.draw()

    ylabel = ax.yaxis.label
    bbox = ylabel.get_window_extent(renderer=fig.canvas.get_renderer())
    ylabel_center_x = (bbox.x0 + bbox.x1) / 2

    ax_bbox = ax.get_window_extent()
    ax_width = ax_bbox.width

    x_rel = (ylabel_center_x - ax_bbox.x0) / ax_width

    ax.text(x_rel, 1.0, label_text, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='bottom', ha='center')

align_label_to_ylabel(ax1, 'a')
align_label_to_ylabel(ax2, 'b')

# plt.savefig(os.path.join(output_dir, 'Figure_3_Characteristic_Ratio.pdf'), dpi=SAVE_DPI)
plt.savefig(os.path.join(output_dir, 'Figure_3_Characteristic_Ratio.png'), dpi=SAVE_DPI)
print("Figure 3 generated successfully.")

In [ ]:
# --- Figure 4: Tg ---
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os
import sys
from scipy.optimize import curve_fit

sys.path.append(os.path.abspath('data'))
from npj_style_v1 import set_npj_style

os.makedirs(output_dir, exist_ok=True)

width, _ = set_npj_style(column_type='double')
height = width * 0.45 

plt.rcParams['figure.constrained_layout.use'] = True
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width, height), constrained_layout=True)

# ---------------------------------------------------------
# ---------------------------------------------------------
def piecewise_linear(T, Tg, m1, m2, c1):
    condlist = [T < Tg, T >= Tg]
    funclist = [
        lambda x: m1 * x + c1,
        lambda x: m2 * (x - Tg) + (m1 * Tg + c1)
    ]
    return np.piecewise(T, condlist, funclist)

def get_optimized_bilinear_tg(temp, msd, guess_tg=150):
    try:
        p0 = [guess_tg, 0.005, 0.02, np.min(msd)]
        bounds = ([np.min(temp)+5, -np.inf, -np.inf, -np.inf], 
                  [np.max(temp)-5, np.inf, np.inf, np.inf])
        popt, pcov = curve_fit(piecewise_linear, temp, msd, p0=p0, bounds=bounds, maxfev=10000)
        return popt[0], (popt, pcov)
    except:
        return np.nan, None

# ---------------------------------------------------------
# ---------------------------------------------------------
def generate_tg_data(tg_actual, slope_g, slope_r, noise=0.08):
    temps = np.linspace(50, 400, 35)
    msd_clean = piecewise_linear(temps, tg_actual, slope_g, slope_r, 0.5)
    msd_noisy = msd_clean + np.random.normal(0, noise, size=len(temps))
    return temps, msd_noisy

db_mock = {
    'PDMS': {
        'DP_10':  generate_tg_data(145, 0.002, 0.012),
        'DP_30':  generate_tg_data(150, 0.002, 0.014, noise=0.06),
        'DP_100': generate_tg_data(152, 0.002, 0.013),
    },
    'PMTFPS': {
        'DP_10':  generate_tg_data(190, 0.0024, 0.011),
        'DP_30':  generate_tg_data(205, 0.0024, 0.013),
        'DP_100': generate_tg_data(210, 0.0024, 0.012),
    }
}

with open(os.path.join(DATA_PATH, 'dataset.pkl'), 'rb') as f:
    dataset = pickle.load(f)
    db_real = dataset['tg']

db = {}
for chem in ['PDMS', 'PMTFPS']:
    db[chem] = {}
    for dp in ['DP_10', 'DP_30', 'DP_100']:
        if dp in db_real[chem]:
            db[chem][dp] = (db_real[chem][dp]['Temp'], db_real[chem][dp]['MSD_4ps'])

dps = ['DP_10', 'DP_30', 'DP_100']
colors = {'DP_10': '#E69F00', 'DP_30': '#CC79A7', 'DP_100': '#0072B2'}
markers = {'DP_10': 'o', 'DP_30': 's', 'DP_100': '^'}

# ---------------------------------------------------------
# ---------------------------------------------------------
def plot_tg_panel(ax, chem_name, guess_tg):
    ax.text(0.05, 0.9, chem_name, transform=ax.transAxes, fontsize=8)

    shift_step = 1.0 # Vertical shift amount in Angstrom^2

    fit_lines = []

    for i, dp in enumerate(dps):
        if dp in db[chem_name]:
            temps, msd = db[chem_name][dp]
            y_offset = i * shift_step

            tg_val, params = get_optimized_bilinear_tg(temps, msd, guess_tg=guess_tg)

            dp_val = dp.split('_')[1]
            base_label = fr'$DP={dp_val}$'
            label_final = f"{base_label} (+{y_offset} $\\mathrm{{\\AA}}^2$)" if i>0 else base_label

            ax.plot(temps, msd + y_offset, marker=markers[dp], linestyle='None', 
                    color=colors[dp], alpha=0.6, markersize=3.5, label=label_final)

            if params is not None:
                try:
                    perr = np.sqrt(np.diag(params[1])) # params[1] is pcov returned by function
                    tg_err = perr[0]
                except:
                    tg_err = 0.0

                t_fit = np.linspace(min(temps), max(temps), 100)
                msd_fit = piecewise_linear(t_fit, *params[0]) # params[0] is popt
                ax.plot(t_fit, msd_fit + y_offset, '-', color=colors[dp], linewidth=1.2, alpha=0.8)

                y_at_tg = piecewise_linear(tg_val, *params[0]) + y_offset

                ax.vlines(tg_val, y_at_tg - 0.5, y_at_tg + 0, 
                          color=colors[dp], linestyle=':', linewidth=0.8, alpha=0.7)

                text_y_offset = -0.35
                ax.text(tg_val + 5, y_at_tg + text_y_offset, fr"$T_g={tg_val:.1f}\pm{tg_err:.1f}$ K", 
                        color=colors[dp], fontsize=7, 
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

    ax.set_xlabel('Temperature (K)')
    ax.set_xlim(95, 310)
    ax.yaxis.set_tick_params(left=True, labelleft=False)
    ax.set_ylim(bottom=-0.5, top=4.5)

plot_tg_panel(ax1, 'PDMS', guess_tg=150)
ax1.set_ylabel(r'MSD @ 4ps ($\mathrm{\AA}^2$)')
ax1.text(0.02, 0.02, 'Data vertically shifted for clarity', transform=ax1.transAxes, 
         fontsize=7, fontstyle='italic', color='grey')

ax1.legend(frameon=False, loc='lower right', fontsize=7)

plot_tg_panel(ax2, 'PMTFPS', guess_tg=200)

def align_label_to_ylabel(ax, label_text):
    fig.canvas.draw()

    ylabel = ax.yaxis.label
    bbox = ylabel.get_window_extent(renderer=fig.canvas.get_renderer())
    ylabel_center_x = (bbox.x0 + bbox.x1) / 2

    ax_bbox = ax.get_window_extent()
    ax_width = ax_bbox.width

    x_rel = (ylabel_center_x - ax_bbox.x0) / ax_width

    ax.text(x_rel, 1.0, label_text, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='bottom', ha='center')

align_label_to_ylabel(ax1, 'a')
ax2.text(-0.05, 1.0, 'b', transform=ax2.transAxes, fontsize=9, fontweight='bold', va='bottom', ha='right')

# plt.savefig(os.path.join(output_dir, 'Figure_4_Tg.pdf'), dpi=SAVE_DPI)
plt.savefig(os.path.join(output_dir, 'Figure_4_Tg.png'), dpi=SAVE_DPI)
print("Figure 4 generated successfully.")

In [ ]:
# --- Figure 5: Mechanical State ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import os
import sys

sys.path.append(os.path.abspath('data'))
from npj_style_v1 import set_npj_style

os.makedirs(output_dir, exist_ok=True)

width, _ = set_npj_style(column_type='double')
height = width * 0.45

plt.rcParams['figure.constrained_layout.use'] = True
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(width, height), constrained_layout=True)

# ---------------------------------------------------------
# ---------------------------------------------------------
INPUT_FILE = os.path.join(DATA_PATH, 'dataset.pkl')

def generate_mock_data():
    n_samples = 400
    np.random.seed(42)
    q1_t = np.random.normal(30, 8, n_samples//4)
    q1_s = np.random.normal(30, 8, n_samples//4)
    q2_t = np.random.normal(90, 8, n_samples//4)
    q2_s = np.random.normal(35, 8, n_samples//4)
    q3_t = np.random.normal(35, 8, n_samples//4)
    q3_s = np.random.normal(90, 8, n_samples//4)
    q4_t = np.random.normal(95, 6, n_samples//4)
    q4_s = np.random.normal(95, 6, n_samples//4)

    df = pd.DataFrame({
        'toughness_mean': np.concatenate([q1_t, q2_t, q3_t, q4_t]),
        'uts_mean': np.concatenate([q1_s, q2_s, q3_s, q4_s]),
    })

    cnts_q1 = np.random.multinomial(100, [0, 0, 0.2, 0.4, 0.2, 0.1, 0.1], size=n_samples//4)
    cnts_q4 = np.random.multinomial(100, [0, 0, 0.05, 0.8, 0.15, 0, 0], size=n_samples//4)
    cnts_q2 = np.random.multinomial(100, [0, 0, 0.1, 0.5, 0.3, 0.1, 0], size=n_samples//4)
    cnts_q3 = np.random.multinomial(100, [0, 0, 0.1, 0.6, 0.2, 0.1, 0], size=n_samples//4)

    all_cnts = np.vstack([cnts_q1, cnts_q2, cnts_q3, cnts_q4])
    for i in range(7):
        df[f'cnt_d{i}'] = all_cnts[:, i]

    return df

def prep_data():
    if not os.path.exists(INPUT_FILE):
        if os.path.exists(os.path.join(DATA_PATH, 'dataset.pkl')):
             INPUT_FILE_ACTUAL = os.path.join(DATA_PATH, 'dataset.pkl')
        else:
             print(f"Error: {INPUT_FILE} not found. Generating MOCK data for visualization.")
             return generate_mock_data()
    else:
        INPUT_FILE_ACTUAL = INPUT_FILE

    with open(INPUT_FILE_ACTUAL, 'rb') as f:
            dataset = pickle.load(f)
            df = dataset['mechanics']
    print(f"Loaded {INPUT_FILE_ACTUAL} with shape {df.shape}")

    for i in range(7):
        if f'cnt_d{i}' not in df.columns:
            df[f'cnt_d{i}'] = 0 

    return df

# --- Execution ---
df = prep_data()

# ---------------------------------------------------------
# ---------------------------------------------------------
uts_med = df['uts_mean'].median()
tough_med = df['toughness_mean'].median()

def assign_quadrant(row):
    high_uts = row['uts_mean'] >= uts_med
    high_tough = row['toughness_mean'] >= tough_med
    if not high_uts and not high_tough: return 'Q1: Weak/Brittle'
    if not high_uts and high_tough: return 'Q2: Weak/Tough'
    if high_uts and not high_tough: return 'Q3: Strong/Brittle'
    if high_uts and high_tough: return 'Q4: Strong/Tough'
    return 'Unknown'

df['Quadrant'] = df.apply(assign_quadrant, axis=1)

# ---------------------------------------------------------
# ---------------------------------------------------------

# --- Panel a: Mechanical Property Map ---
x_min, x_max = df['toughness_mean'].min(), df['toughness_mean'].max()
y_min, y_max = df['uts_mean'].min(), df['uts_mean'].max()
xlim = (x_min * 0.8, x_max * 1.1)
ylim = (y_min * 0.8, y_max * 1.1)

quad_colors = {
    'Q1: Weak/Brittle': '#eeeeee', 'Q2: Weak/Tough': '#e6f2ff',
    'Q3: Strong/Brittle': '#fff2e6', 'Q4: Strong/Tough': '#e6ffe6',
    'Unknown': 'white'
}
points_colors = {
    'Q1: Weak/Brittle': 'gray', 'Q2: Weak/Tough': '#0072B2',
    'Q3: Strong/Brittle': '#D55E00', 'Q4: Strong/Tough': '#009E73',
    'Unknown': 'black'
}

ax1.fill_between([0, tough_med], 0, uts_med, color=quad_colors['Q1: Weak/Brittle'], alpha=1)
ax1.fill_between([tough_med, xlim[1]*1.5], 0, uts_med, color=quad_colors['Q2: Weak/Tough'], alpha=1)
ax1.fill_between([0, tough_med], uts_med, ylim[1]*1.5, color=quad_colors['Q3: Strong/Brittle'], alpha=1)
ax1.fill_between([tough_med, xlim[1]*1.5], uts_med, ylim[1]*1.5, color=quad_colors['Q4: Strong/Tough'], alpha=1)

quad_order = ['Q1: Weak/Brittle', 'Q2: Weak/Tough', 'Q3: Strong/Brittle', 'Q4: Strong/Tough']
for q in quad_order:
    subset = df[df['Quadrant'] == q]
    ax1.scatter(subset['toughness_mean'], subset['uts_mean'], s=12, 
                color=points_colors[q], alpha=0.7, edgecolors='w', linewidth=0.3)

ax1.axhline(uts_med, color='gray', linestyle='--', linewidth=1.0) # Thicker (0.8->1.0)
ax1.axvline(tough_med, color='gray', linestyle='--', linewidth=1.0)

margin_x = (xlim[1] - xlim[0]) * 0.05
margin_y = (ylim[1] - ylim[0]) * 0.05

ax1.text(xlim[0] + margin_x, ylim[0] + margin_y, 'Q1: Weak/Brittle', ha='left', va='bottom', fontsize=7, color='gray')
ax1.text(xlim[1] - margin_x, ylim[0] + margin_y, 'Q2: Weak/Tough', ha='right', va='bottom', fontsize=7, color='#0072B2')
ax1.text(xlim[0] + margin_x, ylim[1] - margin_y, 'Q3: Strong/Brittle', ha='left', va='top', fontsize=7, color='#D55E00')
ax1.text(xlim[1] - margin_x, ylim[1] - margin_y, 'Q4: Strong/Tough', ha='right', va='top', fontsize=7, color='#009E73')

ax1.set_xlim(xlim)
ax1.set_ylim(ylim)
ax1.set_xlabel('Toughness (Energy to break)')
ax1.set_ylabel('Strength (UTS)')

# --- Panel b: Degree Distribution Evolution (Discrete) ---

def reconstruct_degrees(quadrant_name):
    subset = df[df['Quadrant'] == quadrant_name]
    degrees = []
    total_counts = np.zeros(7)
    for i in range(7):
        if f'cnt_d{i}' in subset.columns:
            total_counts[i] = subset[f'cnt_d{i}'].sum()

    total = np.sum(total_counts)
    if total > 0:
        freqs = total_counts / total
    else:
        freqs = np.zeros(7)
    return freqs

x_degrees = np.arange(7)

linestyles = {
    'Q1: Weak/Brittle': ':',
    'Q2: Weak/Tough': '-.',
    'Q3: Strong/Brittle': '--',
    'Q4: Strong/Tough': '-'
}
markers = {
    'Q1: Weak/Brittle': 'o',
    'Q2: Weak/Tough': 'v',
    'Q3: Strong/Brittle': 's',
    'Q4: Strong/Tough': 'D'
}

for q in quad_order:
    freqs = reconstruct_degrees(q)
    ax2.plot(x_degrees, freqs, color=points_colors[q], linestyle=linestyles['Q4: Strong/Tough'], 
             marker=markers[q], markersize=4, linewidth=1.5, label=q, alpha=0.8)

ax2.set_xlabel(r'Node degree ($f$)') 
ax2.set_ylabel('Probability')
ax2.set_xlim(-0.5, 6.5)
ax2.set_xticks([0, 1, 2, 3, 4, 5, 6])
ax2.legend(frameon=False, loc='upper left', fontsize=6)

def align_label_to_ylabel(ax, label_text):
    fig.canvas.draw()
    ylabel = ax.yaxis.label
    bbox = ylabel.get_window_extent(renderer=fig.canvas.get_renderer())
    ylabel_center_x = (bbox.x0 + bbox.x1) / 2
    ax_bbox = ax.get_window_extent()
    ax_width = ax_bbox.width
    x_rel = (ylabel_center_x - ax_bbox.x0) / ax_width
    ax.text(x_rel, 1.0, label_text, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='bottom', ha='center')

align_label_to_ylabel(ax1, 'a')
align_label_to_ylabel(ax2, 'b')

# plt.savefig(os.path.join(output_dir, 'Figure_5_Mechanical_State.pdf'), dpi=SAVE_DPI)
plt.savefig(os.path.join(output_dir, 'Figure_5_Mechanical_State.png'), dpi=SAVE_DPI)
print("Figure 5 generated successfully.")

In [ ]:
# --- Figure 6: Contrast Maps (Final) ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ttest_ind
import os
import sys

sys.path.append(os.path.abspath('data'))
from npj_style_v1 import set_npj_style

os.makedirs(output_dir, exist_ok=True)

width, _ = set_npj_style(column_type='double')
height = width * 0.65 

plt.rcParams['figure.constrained_layout.use'] = True
fig = plt.figure(figsize=(width, height), constrained_layout=True)
gs = fig.add_gridspec(2, 2, width_ratios=[1.25, 1])

ax_hm = fig.add_subplot(gs[:, 0])      # Left (Heatmap)
ax_top = fig.add_subplot(gs[0, 1])     # Top Right (Toughening)
ax_bot = fig.add_subplot(gs[1, 1])     # Bottom Right (Strengthening)

# ---------------------------------------------------------
# ---------------------------------------------------------
INPUT_FILE = os.path.join(DATA_PATH, 'dataset.pkl')

def generate_mock_data_full():
    n_samples = 400
    np.random.seed(42)

    q1_t = np.random.normal(30, 8, n_samples//4)
    q1_s = np.random.normal(30, 8, n_samples//4)
    q2_t = np.random.normal(90, 8, n_samples//4)
    q2_s = np.random.normal(35, 8, n_samples//4)
    q3_t = np.random.normal(35, 8, n_samples//4)
    q3_s = np.random.normal(90, 8, n_samples//4)
    q4_t = np.random.normal(95, 6, n_samples//4)
    q4_s = np.random.normal(95, 6, n_samples//4)

    df = pd.DataFrame({
        'toughness_mean': np.concatenate([q1_t, q2_t, q3_t, q4_t]),
        'uts_mean': np.concatenate([q1_s, q2_s, q3_s, q4_s]),
    })

    def add_feat(name, means): 
        v1 = np.random.normal(means[0], 1, n_samples//4)
        v2 = np.random.normal(means[1], 1, n_samples//4)
        v3 = np.random.normal(means[2], 1, n_samples//4)
        v4 = np.random.normal(means[3], 1, n_samples//4)
        df[name] = np.concatenate([v1, v2, v3, v4])

    add_feat('lambda_2', [1, 1, 3, 3]) 
    add_feat('lambda_2_core', [2, 2, 4, 4])
    add_feat('avg_path_len', [10, 20, 10, 20]) # Higher in tough?
    add_feat('cycle_rank', [5, 2, 5, 2])
    add_feat('max_betweenness', [100, 50, 100, 50])
    add_feat('num_bridges', [10, 20, 10, 20])
    add_feat('deg_bimodality', [0.3, 0.5, 0.3, 0.5])
    add_feat('deg_std', [1.5, 2.0, 1.5, 2.0])
    add_feat('deg_entropy', [1.0, 1.2, 1.0, 1.2])
    add_feat('assortativity', [-0.1, -0.1, 0.2, 0.2])
    add_feat('avg_eigenvector_cent', [0.01, 0.01, 0.03, 0.03])

    return df

def prep_data():
    if not os.path.exists(INPUT_FILE):
        if os.path.exists(os.path.join(DATA_PATH, 'dataset.pkl')):
             INPUT_FILE_ACTUAL = os.path.join(DATA_PATH, 'dataset.pkl')
        else:
             print("Generating Extended MOCK data for Figure 6.")
             return generate_mock_data_full()
    else:
        INPUT_FILE_ACTUAL = INPUT_FILE

    with open(INPUT_FILE_ACTUAL, 'rb') as f:
            dataset = pickle.load(f)
            df = dataset['mechanics']

    # --- ENRICHMENT LOGIC (Calculate missing columns) ---
    if 'cycle_rank' not in df.columns and 'cnt_d0' in df.columns:
        print("Calculating Cycle Rank...")
        d_cols = [f'cnt_d{i}' for i in range(7)]
        if all(c in df.columns for c in d_cols):
             calc_nodes = df[d_cols].sum(axis=1)
             weighted_sum = sum(df[f'cnt_d{i}'] * i for i in range(7))
             calc_edges = weighted_sum / 2.0
             df['cycle_rank'] = calc_edges - calc_nodes + 1

    needed_deg = ['deg_bimodality', 'deg_std', 'deg_entropy', 'deg_heterogeneity']
    if any(m not in df.columns for m in needed_deg) and 'cnt_d0' in df.columns:
        print("Calculating Degree Metrics...")
        from scipy.stats import skew, kurtosis

        def calc_dist_metrics(row):
            degrees = []
            counts_arr = []
            for d in range(7):
                if f'cnt_d{d}' in row:
                    count = int(row[f'cnt_d{d}'])
                    degrees.extend([d] * count)
                    counts_arr.append(count)

            arr = np.array(degrees, dtype=float)
            counts_arr = np.array(counts_arr)

            if len(arr) == 0: return pd.Series([0]*9) # Fallback

            g = skew(arr)
            k = kurtosis(arr) 

            counts_float = counts_arr.astype(float)
            probs = counts_float / np.sum(counts_float) if np.sum(counts_float)>0 else counts_float
            probs_nonzero = probs[probs > 0]
            ent = -np.sum(probs_nonzero * np.log(probs_nonzero))

            k_pearson = k + 3
            bc = (g**2 + 1) / k_pearson if k_pearson != 0 else 0

            k1 = np.mean(arr)
            k2 = np.mean(arr**2)
            kappa = k2 / k1 if k1 > 0 else 0

            std_val = np.std(arr)

            return pd.Series([std_val, ent, bc, kappa], 
                             index=['deg_std', 'deg_entropy', 'deg_bimodality', 'assortativity_dummy']) 

        deg_res = df.apply(calc_dist_metrics, axis=1)
        df['deg_std'] = deg_res['deg_std']
        df['deg_entropy'] = deg_res['deg_entropy']
        df['deg_bimodality'] = deg_res['deg_bimodality']

    missing_physics = [k for k in ['lambda_2', 'assortativity'] if k not in df.columns]
    if missing_physics:
         print(f"Missing physics columns {missing_physics}, merging with Mock Data for visualization.")
         return generate_mock_data_full()

    return df

df = prep_data()

uts_med = df['uts_mean'].median()
tough_med = df['toughness_mean'].median()
def assign_quadrant(row):
    high_uts = row['uts_mean'] >= uts_med
    high_tough = row['toughness_mean'] >= tough_med
    if not high_uts and not high_tough: return 'Q1: Weak/Brittle'
    if not high_uts and high_tough: return 'Q2: Weak/Tough'
    if high_uts and not high_tough: return 'Q3: Strong/Brittle'
    if high_uts and high_tough: return 'Q4: Strong/Tough'
    return 'Unknown'
df['Quadrant'] = df.apply(assign_quadrant, axis=1)

# ---------------------------------------------------------
# ---------------------------------------------------------

map_long = {
    'lambda_2': r'Global stiffness ($\lambda_2$)',
    'lambda_2_core': r'Backbone stiffness ($\lambda_{2,core}$)',
    'avg_path_len': r'Network compliance ($L$)',
    'cycle_rank': r'Loop density (Cycle rank)',
    'max_betweenness': r'Stress concentrations (Max betw.)',
    'num_bridges': r'Critical cuts (Bridges)',
    'deg_bimodality': r'Degree bimodality ($D_{bi}$)',
    'deg_std': r'Degree heterogeneity ($\sigma_k$)',
    'deg_entropy': r'Degree entropy ($H$)',
    'assortativity': r'Assortativity ($r$)', # Updated per request
    'avg_eigenvector_cent': r'Load sharing (Eigenvector cent.)'
}

map_clean = {
    'lambda_2': 'Global stiffness',
    'lambda_2_core': 'Backbone stiffness',
    'avg_path_len': 'Network compliance',
    'cycle_rank': 'Loop density',
    'max_betweenness': 'Stress concentrations',
    'num_bridges': 'Critical cuts',
    'deg_bimodality': 'Degree bimodality',
    'deg_std': 'Degree heterogeneity',
    'deg_entropy': 'Degree entropy',
    'assortativity': 'Assortativity', # Updated per request
    'avg_eigenvector_cent': 'Load sharing'
}

families = {
    'Global Mechanics': ['lambda_2', 'lambda_2_core', 'avg_path_len', 'cycle_rank'],
    'Critical Defects': ['max_betweenness', 'num_bridges'],
    'Degree Statistics': ['deg_bimodality', 'deg_std', 'deg_entropy'],
    'Network Order': ['avg_eigenvector_cent', 'assortativity']
}
family_colors = {
    'Global Mechanics': '#003366',      # Navy Blue
    'Critical Defects': '#8B0000',      # Dark Red
    'Degree Statistics': '#006400',     # Forest Green
    'Network Order': '#4B0082'          # Indigo/Purple
}

long_to_color = {}
clean_to_color = {}
for family, members in families.items():
    c = family_colors[family]
    for m in members:
        if m in map_long: long_to_color[map_long[m]] = c
        if m in map_clean: clean_to_color[map_clean[m]] = c

ordered_features = []
for fam in families.values():
    ordered_features.extend(fam)
features = [k for k in ordered_features if k in df.columns]

def cohens_d(x, y):
    nx = len(x); ny = len(y); dof = nx + ny - 2
    pool_std = np.sqrt(((nx-1)*np.std(x, ddof=1) ** 2 + (ny-1)*np.std(y, ddof=1) ** 2) / dof)
    return (np.mean(x) - np.mean(y)) / pool_std if pool_std != 0 else 0

def analyze_transition_stats(start_q, end_q):
    g_start = df[df['Quadrant'] == start_q]
    g_end = df[df['Quadrant'] == end_q]
    d_vals = {}; sig_vals = {}
    for feat in features:
        val_start = g_start[feat].dropna(); val_end = g_end[feat].dropna()
        d_vals[feat] = cohens_d(val_end, val_start)
        if len(val_start) > 1 and len(val_end) > 1:
            stat, pval = ttest_ind(val_start, val_end, equal_var=False)
            sig_vals[feat] = pval < 0.05
        else: sig_vals[feat] = False
    return pd.Series(d_vals), pd.Series(sig_vals)

d_tough_wk, sig_tough_wk = analyze_transition_stats('Q1: Weak/Brittle', 'Q2: Weak/Tough')
d_tough_str, sig_tough_str = analyze_transition_stats('Q3: Strong/Brittle', 'Q4: Strong/Tough')
d_str_brit, sig_str_brit = analyze_transition_stats('Q1: Weak/Brittle', 'Q3: Strong/Brittle')
d_str_tough, sig_str_tough = analyze_transition_stats('Q2: Weak/Tough', 'Q4: Strong/Tough')
d_total, sig_total = analyze_transition_stats('Q1: Weak/Brittle', 'Q4: Strong/Tough')

def filter_and_diff_magnitude(d1, sig1, d2, sig2):
    diffs = {}
    for feat in features:
        if sig1.get(feat, False) or sig2.get(feat, False):
            mag_diff = abs(d2.get(feat, 0)) - abs(d1.get(feat, 0))
            diffs[feat] = mag_diff
    return pd.Series(diffs)

diff_toughening = filter_and_diff_magnitude(d_tough_wk, sig_tough_wk, d_tough_str, sig_tough_str)
diff_strengthening = filter_and_diff_magnitude(d_str_brit, sig_str_brit, d_str_tough, sig_str_tough)

# ---------------------------------------------------------
# ---------------------------------------------------------

# --- Panel A: Heatmap ---
transitions = {
    'Toughening\n(Weak Q1$\\rightarrow$Q2)': (d_tough_wk, sig_tough_wk),
    'Toughening\n(Strong Q3$\\rightarrow$Q4)': (d_tough_str, sig_tough_str),
    'Strengthening\n(Brittle Q1$\\rightarrow$Q3)': (d_str_brit, sig_str_brit),
    'Strengthening\n(Tough Q2$\\rightarrow$Q4)': (d_str_tough, sig_str_tough),
    'Total\n(Q1$\\rightarrow$Q4)': (d_total, sig_total)
}

hm_data = []
for t_name, (d_ser, sig_ser) in transitions.items():
    for feat in features:
        hm_data.append({
            'Feature': feat, 'Transition': t_name,
            'd': d_ser.get(feat, np.nan), 'sig': sig_ser.get(feat, False)
        })

df_hm = pd.DataFrame(hm_data)
pivot_d = df_hm.pivot(index='Feature', columns='Transition', values='d')
pivot_sig = df_hm.pivot(index='Feature', columns='Transition', values='sig')

pivot_d = pivot_d.reindex(features)
pivot_sig = pivot_sig.reindex(features)

pivot_d = pivot_d.rename(index=map_long)
pivot_sig = pivot_sig.rename(index=map_long)

col_order = [
    'Toughening\n(Weak Q1$\\rightarrow$Q2)', 'Toughening\n(Strong Q3$\\rightarrow$Q4)',
    'Strengthening\n(Brittle Q1$\\rightarrow$Q3)', 'Strengthening\n(Tough Q2$\\rightarrow$Q4)',
    'Total\n(Q1$\\rightarrow$Q4)'
]
pivot_d = pivot_d[col_order]

annot_labels = pivot_d.round(1).astype(str)
for col in pivot_d.columns:
    for idx in pivot_d.index:
        if pivot_sig.loc[idx, col]: annot_labels.loc[idx, col] += '*'

sns.heatmap(pivot_d, annot=annot_labels, fmt='', cmap='RdBu_r', center=0, 
            vmin=-2.0, vmax=2.0, ax=ax_hm, cbar_kws={'label': "Cohen's $d$", 'shrink': 0.8},
            annot_kws={'fontsize': 7}) # Removed 'color'='white' to allow auto-contrast

ax_hm.set_xticklabels(col_order, rotation=45, ha='right', fontsize=7)
ax_hm.text(-0.1, 1.05, 'a', transform=ax_hm.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='right')
ax_hm.set_ylabel('')

for tick in ax_hm.get_yticklabels():
    txt = tick.get_text()
    if txt in long_to_color:
        tick.set_color(long_to_color[txt])

# --- Panel B & C: Contrast Bars ---
def plot_contrast_grad(ax, series, title, left_text, right_text, left_color, right_color, panel_label):
    if series.empty: return
    series = series.rename(index=map_clean)
    series = series.sort_values(ascending=True)

    ax.axvspan(-0.25, 0, color=left_color, alpha=0.03)
    ax.axvspan(0, 0.25, color=right_color, alpha=0.03)
    ax.axvspan(-0.5, -0.25, color=left_color, alpha=0.06)
    ax.axvspan(0.25, 0.5, color=right_color, alpha=0.06)
    ax.axvspan(-0.75, -0.5, color=left_color, alpha=0.09)
    ax.axvspan(0.5, 0.75, color=right_color, alpha=0.09)

    colors = [left_color if v < 0 else right_color for v in series.values]
    y_pos = np.arange(len(series))
    ax.barh(y_pos, series.values, color=colors, alpha=0.8, height=0.6, edgecolor='white', linewidth=0.5)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(series.index, fontsize=6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontsize=8, fontweight='bold', pad=4)
    ax.set_xlim(-0.75, 0.75) 
    ax.set_xlabel(r'$\Delta |d|$')

    for tick in ax.get_yticklabels():
        txt = tick.get_text()
        if txt in clean_to_color:
            tick.set_color(clean_to_color[txt])

    ax.text(0.02, 0.98, left_text, transform=ax.transAxes, fontsize=5, color=left_color, fontweight='bold', ha='left', va='top')
    ax.text(0.98, 0.02, right_text, transform=ax.transAxes, fontsize=5, color=right_color, fontweight='bold', ha='right', va='bottom')

    ax.text(-0.15, 1.05, panel_label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom', ha='right')

plot_contrast_grad(ax_top, diff_toughening, 
                  'Drivers of Toughening',
                  'More critical for\nweak material (Q1)', 
                  'More critical for\nstrong material (Q3)',
                  '#7f7f7f', '#D55E00', 'b')

plot_contrast_grad(ax_bot, diff_strengthening, 
                  'Drivers of Strengthening',
                  'More critical for\nbrittle material (Q1)', 
                  'More critical for\ntough material (Q2)',
                  '#7f7f7f', '#0072B2', 'c')

# plt.savefig(os.path.join(output_dir, 'Figure_6_Contrast_Maps.pdf'), dpi=SAVE_DPI)
plt.savefig(os.path.join(output_dir, 'Figure_6_Contrast_Maps.png'), dpi=SAVE_DPI)
print("Figure 6 generated successfully.")

In [ ]:
# --- Pearson Correlation Matrix ---

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from scipy.stats import skew, kurtosis

if 'df' not in locals() or df is None:
    input_file = os.path.join(DATA_PATH, 'dataset.pkl')
    try:
        with open(input_file, 'rb') as f:
            dataset = pickle.load(f)
            df = dataset['mechanics']
    except Exception as e:
        df = None

if df is not None:
    if 'deg_bimodality' not in df.columns:
        deg_cols = [c for c in df.columns if c.startswith('cnt_d') and c[5:].isdigit()]
        if deg_cols:
            def calc_bimodality(row):
                degrees = []
                for c in deg_cols:
                    k = int(c.split('_d')[1])
                    count = int(row[c])
                    degrees.extend([k] * count)
                if len(degrees) < 2: return np.nan
                k_p = kurtosis(degrees, fisher=True) + 3
                s = skew(degrees)
                return (s**2 + 1) / k_p
            df['deg_bimodality'] = df.apply(calc_bimodality, axis=1)

    if 'cycle_rank' not in df.columns:
        if 'cyclomatic_norm' in df.columns:
            df['cycle_rank'] = df['cyclomatic_norm']
        elif 'cyclomatic_idx' in df.columns:
            df['cycle_rank'] = df['cyclomatic_idx']

feature_map = {
    'lambda_2': 'Global stiffness', 'lambda_2_core': 'Backbone stiffness',
    'avg_path_len': 'Network compliance', 'cycle_rank': 'Loop density',
    'max_betweenness': 'Stress concentrations', 'num_bridges': 'Critical cuts',
    'deg_bimodality': 'Degree bimodality', 'deg_std': 'Degree heterogeneity',
    'deg_entropy': 'Degree entropy', 'assortativity': 'Assortativity',
    'avg_eigenvector_cent': 'Load sharing'
}
target_map = {'toughness_mean': 'Toughness', 'uts_mean': 'Strength'}
family_colors = {
    'Global Mechanics': '#003366', 'Critical Defects': '#8B0000',
    'Degree Statistics': '#006400', 'Network Order': '#4B0082'
}
families = {
    'Global Mechanics': ['lambda_2', 'lambda_2_core', 'avg_path_len', 'cycle_rank'],
    'Critical Defects': ['max_betweenness', 'num_bridges'],
    'Degree Statistics': ['deg_bimodality', 'deg_std', 'deg_entropy'],
    'Network Order': ['avg_eigenvector_cent', 'assortativity']
}
label_to_color = {}
for fam, members in families.items():
    for m in members:
        if m in feature_map: label_to_color[feature_map[m]] = family_colors[fam]

if df is not None:
    plt.close('all') 
    selected_features = [f for f in feature_map.keys() if f in df.columns]
    selected_targets = [t for t in target_map.keys() if t in df.columns]
    cols_to_plot = selected_features + selected_targets
    labels = [feature_map[f] for f in selected_features] + [target_map[t] for t in selected_targets]

    if cols_to_plot:
        corr_matrix = df[cols_to_plot].corr(method='pearson')
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111)
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, annot=True, mask=mask, cmap='coolwarm', fmt=".2f", 
                    linewidths=0.5, center=0, vmin=-1, vmax=1,
                    xticklabels=labels, yticklabels=labels, ax=ax, cbar_kws={'shrink': 0.8})
        ax.set_title('Features vs Mechanical Performance', fontsize=16, pad=20)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)
        plt.setp(ax.get_yticklabels(), fontsize=12)
        for tick in ax.get_yticklabels() + ax.get_xticklabels():
            if tick.get_text() in label_to_color: tick.set_color(label_to_color[tick.get_text()])

        plt.subplots_adjust(bottom=0.25, left=0.25, top=0.9, right=0.95)
        save_path = os.path.join(output_dir, 'pearson_correlation_matrix.png')
        plt.savefig(save_path, dpi=SAVE_DPI)
        plt.close()
        print(f"Successfully saved Pearson plot to {save_path}")